# Single-Channel Response Curve & Saturation Modeling: Practitioner's Template

**Author:** [Ryan Duecker](mailto:ryanduecker@google.com) | **Module:** `tippingpoint`

This notebook provides a **production-ready boilerplate template** for growth marketers, media scientists, and econometrics practitioners. 

Unlike capability-showcase tutorials, this template is structured as an operational, plug-and-play workflow designed to take raw campaign spend and return data through:
1. **Practitioner Configuration & Data Ingestion:** Centralized configuration parameters (target mROAS hurdle rate, spend baseline, adstock bounds) and data mapping.
2. **Single-Channel Curve Fitting:** Hill saturation and adstock carryover fitting with statistical uncertainty intervals.
3. **Causal Incrementality Validation & Calibration:** Validating observational curves against lift tests (holdouts / geo-experiments) or calibrating with Bayesian MCMC.
4. **Diagnostic Fit Evaluation & Decision Callouts:** Automated heuristic health-checks flagging low explanatory power, high MAPE, residual bias, parameter extrapolation, and carryover anomalies.
5. **Inflection Points & Headroom Decisioning:** Peak Efficiency ($f''(x) = 0$), Point of Diminishing Returns ($f'(x) = \text{Target mROAS}$), and Optimal Scaling Zone.
6. **Module Visualizations & Matplotlib Customization:** Out-of-the-box response curves with direct interception of the `matplotlib.figure.Figure` object for custom annotations, executive branding, and multi-panel reporting.
7. **Forward-Looking Capacity Projection:** Overcoming observational saturation ceilings by projecting capacity unlocks using vertical benchmark modifiers (funnel transitions, Reach & Frequency penetration, and damped multi-driver synergies).

## 1. Setup & Central Configuration

Configure your media channel parameters, financial hurdle rates, and modeling options below. This central configuration block governs all downstream curve fitting, diagnostic callouts, and scaling evaluations.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from tippingpoint import (
    MarketingReturnCurve,
    CapacityProjector,
    ProjectedReturnCurve,
    CapacityMultipliers,
)

# ==============================================================================
# 🛠️ PRACTITIONER CONFIGURATION: Set your channel parameters & hurdle rates
# ==============================================================================

# 1. Channel Identity & Economics
CHANNEL_NAME = "Paid Search / Performance Video"
TARGET_MROAS = 1.15            # Target marginal ROAS hurdle rate (or 1.0 / Target CPA)
CURRENT_DAILY_SPEND = 14_000.0 # Current operational daily spend level

# 2. Curve Fitting Engine & Adstock Settings
# Method options: 'auto', 'frequentist' (NLS with SEs & CIs), 'bayesian' (MCMC with posteriors), 'gradient_descent'
FIT_METHOD = "frequentist"
ADSTOCK_TYPE = "bounded"       # 'none', 'bounded', 'fixed', 'free'
ADSTOCK_BOUNDS = (1.0, 14.0)   # Half-life search bounds in days (min_days, max_days)
ADSTOCK_FIXED_DAYS = 3.5       # Used only if ADSTOCK_TYPE == 'fixed'
FIT_BASELINE = True            # Whether to estimate unadvertised organic baseline demand
CONFIDENCE_LEVEL = 0.95        # Confidence / Credible interval width

# 3. Optional Incrementality Experiment Data (Set to None if no test available)
# Single experiment or list of dicts: {'spend': float, 'lift': float, 'se': float, 'name': str}
EXPERIMENTS = [
    {"name": "Geo_Holdout_Q1", "spend": 10000.0, "lift": 12500.0, "se": 950.0},
    {"name": "Lift_Test_Q2",   "spend": 18000.0, "lift": 23200.0, "se": 1400.0},
]

## 2. Data Ingestion & Preparation

Import your time-series media spend and response dataset. The dataset can be at daily or weekly granularity. 

> [!TIP]
> **Data Preparation Best Practices:**
> - Ensure **Spend** and **Return** (revenue, conversions, or pipeline value) are aligned on the same calendar timestamps.
> - When modeling multiple sub-campaigns or tactics within the same channel, aggregate them to total channel daily/weekly totals before fitting.
> - If you already have your own CSV, uncomment the `pd.read_csv(...)` line below. A realistic synthetic dataset is generated by default for out-of-the-box execution.

In [ ]:
# ------------------------------------------------------------------------------
# DATA INGESTION: Swap this block with your production CSV / SQL extract
# Example: df = pd.read_csv("path_to_campaign_data.csv")
#          spends = df["spend"].values
#          returns = df["return"].values
# ------------------------------------------------------------------------------

# Generate a realistic 120-day benchmark dataset with organic baseline & adstock carryover
np.random.seed(42)
n_days = 120
date_range = pd.date_range(start="2026-01-01", periods=n_days, freq="D")

# Daily spend variations ($2.5k to $25k) with weekly pacing dynamics
base_spend = np.random.uniform(2500, 25000, size=n_days)
day_of_week = date_range.dayofweek.values
spend_series = base_spend * (1.0 + 0.12 * np.sin(day_of_week / 7.0 * 2 * np.pi))

# True underlying DGP (Data Generating Process) for demonstration
theta_true = 0.35
s_adstocked_true = np.zeros(n_days)
for t in range(n_days):
  prev = s_adstocked_true[t - 1] if t > 0 else 0.0
  s_adstocked_true[t] = spend_series[t] + theta_true * prev

beta_true = 48000.0
alpha_true = 1.60
k_true = 16000.0
baseline_true = 5000.0

y_true = baseline_true + beta_true * (s_adstocked_true ** alpha_true) / (k_true ** alpha_true + s_adstocked_true ** alpha_true)
noise = np.random.normal(0, 1200.0, size=n_days)
return_series = np.maximum(0, y_true + noise)

df = pd.DataFrame({
    "date": date_range,
    "spend": np.round(spend_series, 2),
    "return": np.round(return_series, 2)
})

spends = df["spend"].values
returns = df["return"].values

print(f"Data Loaded: {len(df)} daily observations for [{CHANNEL_NAME}]")
print(f"Spend Range:  ${df['spend'].min():,.2f} to ${df['spend'].max():,.2f} (Mean: ${df['spend'].mean():,.2f})")
print(f"Return Range: ${df['return'].min():,.2f} to ${df['return'].max():,.2f} (Mean: ${df['return'].mean():,.2f})")
df.head(5)

## 3. Single-Channel Curve Fitting

We fit the non-linear Hill saturation response curve and adstock decay to our historical data.

### Mathematical Formulation:
$$Return_t = \beta_0 + \frac{\beta \cdot S_{t,\text{adstocked}}^\alpha}{K^\alpha + S_{t,\text{adstocked}}^\alpha}$$

*   **$\beta$ (Beta - Capacity):** Maximum achievable incremental return.
*   **$\alpha$ (Alpha - Shape):** Learning elasticity. $\alpha > 1$ represents an S-curve (initial warm-up phase); $\alpha \le 1$ represents a concave C-curve (immediate diminishing returns).
*   **$K$ (Half-Saturation):** Effective spend needed to achieve 50% of peak incremental return capacity $\beta$.
*   **$\beta_0$ (Baseline):** Non-media organic return generated even at zero spend.
*   **$\theta$ (Adstock Memory):** Daily retention rate converting raw spend into cumulative carryover spend: $S_{t,\text{adstocked}} = S_t + \theta \cdot S_{t-1,\text{adstocked}}$. Half-life is $t_{1/2} = -\frac{\ln(2)}{\ln(\theta)}$.

In [ ]:
# ------------------------------------------------------------------------------
# 3. FIT RESPONSE CURVE
# ------------------------------------------------------------------------------
model = MarketingReturnCurve.fit(
    spend_array=spends,
    return_array=returns,
    channel_name=CHANNEL_NAME,
    method=FIT_METHOD,
    adstock_type=ADSTOCK_TYPE,
    adstock_bounds=ADSTOCK_BOUNDS,
    adstock_fixed_days=ADSTOCK_FIXED_DAYS,
    fit_baseline=FIT_BASELINE,
    confidence_level=CONFIDENCE_LEVEL
)

# Display fitted parameters
summary = model.summary()
p = summary["parameters"]

print("\n" + "=" * 70)
print(f"  FITTED PARAMETERS SUMMARY: {CHANNEL_NAME}")
print("=" * 70)
print(f"  • Beta (Max Incremental Capacity β): ${p['beta']:>12,.2f}")
print(f"  • Alpha (Learning Curve Shape α):    {p['alpha']:>13.4f} ({'S-Curve: Warm-up period exists' if p['alpha'] > 1 else 'C-Curve: Diminishing returns from dollar 1'})")
print(f"  • Half-Saturation (K):               ${p['K']:>12,.2f}")
print(f"  • Organic Baseline Return (β₀):      ${p['baseline']:>12,.2f}")
print(f"  • Adstock Retention Rate (θ):        {p['theta']:>13.4f} (Carryover Half-Life: {p['adstock_half_life_days']:.2f} days)")

if "standard_errors" in summary and summary["standard_errors"] is not None:
  se = summary["standard_errors"]
  ci = summary.get("confidence_intervals", {})
  print("\n  Parameter Statistical Uncertainty (95% Confidence Intervals):")
  for k_param in ["beta", "alpha", "K", "baseline", "theta"]:
    if k_param in se:
      ci_low, ci_high = ci.get(k_param, (np.nan, np.nan))
      print(f"    - {k_param:<9}: SE = {se[k_param]:>10,.2f} | 95% CI: [{ci_low:>10,.2f}, {ci_high:>10,.2f}]")
print("=" * 70)

## 4. Incrementality Experiment Validation & Calibration (Optional / Modular)

Observational curves can suffer from selection bias or baseline demand confounding. Grounding or validating response curves against causal experiments (e.g., Geo-experiments, Conversion Lift tests, Audience Holdouts) ensures executive confidence.

### Two Practitioner Workflows:
1. **Post-Hoc Validation (`validate_experiments`):** Evaluate whether your fitted observational curve aligns with experimental ground-truth results. Generates Z-scores, 95% CI coverage, reduced $\chi^2$, and an overall alignment verdict (`EXCELLENT`, `ALIGNED`, or `MISALIGNED`).
2. **Bayesian Calibration (`fit_bayesian`):** Re-fit the curve using experimental lift measurements as Bayesian likelihood anchors, aligning the curve with causal ground truth while estimating saturation shape from observational variance.

In [ ]:
# ------------------------------------------------------------------------------
# 4. EXPERIMENT VALIDATION & CALIBRATION
# ------------------------------------------------------------------------------

if EXPERIMENTS is not None and len(EXPERIMENTS) > 0:
  print("--- Step 4A: Post-Hoc Incrementality Validation ---")
  # Attach experiments to the model
  model.attach_experiments(EXPERIMENTS)
  
  # Run validation (evaluates Z-scores, CI coverage %, reduced chi^2, and verdict)
  val_report = model.validate_experiments(spend_is_raw=True, verbose=True)
  
  # Check if calibration is warranted
  if val_report["verdict"] == "MISALIGNED":
    print("\n⚠️ WARNING: Observational model is MISALIGNED with causal lift experiments.")
    print("Recommendation: Re-fit using Bayesian MCMC calibration below to anchor the curve to causal truth.\n")
  else:
    print(f"\n✅ SUCCESS: Observational curve is {val_report['verdict']} with experimental lift studies.")

  # Optional Step 4B: Re-fit with Bayesian MCMC Calibration (demonstration)
  # Uncomment the lines below to enforce Bayesian calibration:
  # calibrated_model = MarketingReturnCurve.fit_bayesian(
  #     spend_array=spends,
  #     return_array=returns,
  #     channel_name=f"{CHANNEL_NAME} (Calibrated)",
  #     adstock_type=ADSTOCK_TYPE,
  #     adstock_bounds=ADSTOCK_BOUNDS,
  #     calibration_experiments=EXPERIMENTS,
  #     fit_baseline=FIT_BASELINE,
  #     n_samples=2000,
  #     burn_in=500
  # )
  # model = calibrated_model  # Replace working model with calibrated version
else:
  print("No incrementality experiments configured. Proceeding with observational model.")

## 5. Diagnostic Fit Evaluation & Automated Decision Callouts

Evaluating goodness-of-fit is essential before using curves for capital allocation. Media data is inherently noisy, but statistical metrics reveal whether a curve is reliable, overfitted, biased, or extrapolating into ungrounded spend territories.

### Statistical Fit Criteria & Rules-of-Thumb:
1. **$R^2$ / Adjusted $R^2$ (Explanatory Power):**
   - $R^2 \ge 0.80$: Strong fit — media spend cleanly explains response variation.
   - $0.60 \le R^2 < 0.80$: Moderate / Acceptable fit — standard for noisy daily media performance.
   - $R^2 < 0.60$: **Poor fit** — media spend explains less than 60% of variance. Signals unmodeled confounding (promotions, price changes, competitor pressure, seasonality).
2. **Mean Absolute Percentage Error (MAPE):**
   - $\text{MAPE} \le 15\%$: High commercial precision.
   - $15\% < \text{MAPE} \le 25\%$: Acceptable commercial accuracy.
   - $\text{MAPE} > 25\%$: **High relative error** — outliers or structural breaks present.
3. **Residual Bias ($|\text{Mean Residual}| / \text{RMSE}$):**
   - Systematic deviations from 0 indicate omitted baseline demand ($\beta_0$) or mis-specified adstock decay ($\theta$).
4. **Parameter Plausibility & Extrapolation Guardrails:**
   - **$K$ vs Max Spend:** If $K > 2 \times \max(\text{Spend})$, the half-saturation point is unobserved, meaning high-budget forecasts rely heavily on extrapolation.
   - **$\alpha$ Plausibility:** $\alpha > 3.5$ indicates an unnaturally steep step function. $\alpha > 1$ confirms an S-curve (learning threshold), while $\alpha \le 1$ indicates a C-curve.
   - **Adstock $\theta$:** Carryover half-life $> 14$ days on performance digital channels suggests non-stationarity or omitted seasonality rather than true brand memory.

In [ ]:
# ------------------------------------------------------------------------------
# 5. GOODNESS-OF-FIT EVALUATION & AUTOMATED DIAGNOSTIC CALLOUTS
# ------------------------------------------------------------------------------

# 1. Compute statistical goodness-of-fit metrics
metrics = model.evaluate_fit(verbose=True)

# 2. Automated Diagnostic Decision Engine & Health-Check Callouts
def run_fit_diagnostics(model, metrics, spend_data, return_data, experiments=None):
  """Evaluates statistical fit and parameter plausibility, printing structured callouts."""
  callouts = []
  warnings_count = 0
  
  # --- Check 1: Explanatory Power (R^2 & Adj R^2) ---
  r2 = metrics["r_squared"]
  adj_r2 = metrics["adj_r_squared"]
  r2_gap = r2 - adj_r2
  
  if r2 >= 0.80:
    callouts.append(("✅", "Explanatory Power (R²)", f"{r2:.4f} (Strong)", "Media spend reliably captures the underlying variance."))
  elif r2 >= 0.60:
    callouts.append(("🟡", "Explanatory Power (R²)", f"{r2:.4f} (Moderate)", "Fit is acceptable for commercial media data; some unexplained noise remains."))
  else:
    warnings_count += 1
    callouts.append(("⚠️", "Explanatory Power (R²)", f"{r2:.4f} (POOR FIT)", 
                     "Media spend explains <60% of variance. Check for promotions, seasonality, macroeconomic shifts, or tracking issues."))

  if r2_gap > 0.05:
    warnings_count += 1
    callouts.append(("⚠️", "Overfitting Gap (R² - Adj R²)", f"{r2_gap:.4f}", 
                     "Sample size may be too small relative to fitted parameters (beta, alpha, K, baseline, theta)."))

  # --- Check 2: Relative Error (MAPE) ---
  mape = metrics["mape"]
  if mape <= 15.0:
    callouts.append(("✅", "Relative Error (MAPE)", f"{mape:.2f}% (High Accuracy)", "Mean relative error is within strict econometric standards."))
  elif mape <= 25.0:
    callouts.append(("🟡", "Relative Error (MAPE)", f"{mape:.2f}% (Commercial Grade)", "Standard variance band for daily media performance."))
  else:
    warnings_count += 1
    callouts.append(("⚠️", "Relative Error (MAPE)", f"{mape:.2f}% (HIGH ERROR)", 
                     "Predictions deviate substantially from actuals. Look for extreme spend outliers or structural breaks."))

  # --- Check 3: Residual Bias ---
  res_mean = metrics["residual_mean"]
  res_std = metrics["residual_std"]
  bias_ratio = abs(res_mean) / (res_std + 1e-9)
  
  if bias_ratio < 0.10:
    callouts.append(("✅", "Residual Distribution", f"Mean: {res_mean:+.2f} (Unbiased)", "Errors are well-centered around zero without systematic drift."))
  else:
    warnings_count += 1
    callouts.append(("⚠️", "Residual Bias", f"Mean: {res_mean:+.2f} (Biased)", 
                     "Systematic prediction bias detected. If baseline is 0, consider setting fit_baseline=True or tuning adstock bounds."))

  # --- Check 4: Parameter Plausibility & Extrapolation ---
  max_obs_spend = np.max(spend_data)
  min_obs_spend = np.min(spend_data)
  k_val = model.K
  alpha_val = model.alpha
  theta_val = model.theta
  
  # Half-saturation K extrapolation check
  if k_val > 2.0 * max_obs_spend:
    warnings_count += 1
    callouts.append(("⚠️", "Half-Saturation (K) Extrapolation", f"K = ${k_val:,.0f} vs Max Spend ${max_obs_spend:,.0f}", 
                     "Half-saturation point is far beyond observed spend range. Forecasts near K are ungrounded extrapolations."))
  elif k_val < 0.2 * min_obs_spend:
    warnings_count += 1
    callouts.append(("⚠️", "Premature Saturation", f"K = ${k_val:,.0f} vs Min Spend ${min_obs_spend:,.0f}", 
                     "Channel is already at saturation across historical data. Decreasing spend may improve efficiency."))
  else:
    callouts.append(("✅", "Half-Saturation (K) Plausibility", f"${k_val:,.0f}", "Half-saturation point is well-bracketed by historical observations."))

  # Alpha shape check
  if alpha_val > 3.5:
    warnings_count += 1
    callouts.append(("⚠️", "Alpha Shape (α)", f"{alpha_val:.2f} (Unusually Steep)", 
                     "Extremely sharp S-curve threshold. Verify whether data contains an artificial gating artifact."))
  elif alpha_val > 1.0:
    callouts.append(("✅", "Alpha Shape (α)", f"{alpha_val:.2f} (S-Curve)", 
                     "Initial learning/warm-up phase confirmed. Peak efficiency exists at inflection spend."))
  else:
    callouts.append(("✅", "Alpha Shape (α)", f"{alpha_val:.2f} (C-Curve)", 
                     "Immediate concave diminishing returns from dollar 1. Minimal marginal cost is at $0."))

  # Adstock carryover check
  half_life = model.summary()["parameters"]["adstock_half_life_days"]
  if half_life > 14.0:
    warnings_count += 1
    callouts.append(("⚠️", "Adstock Half-Life", f"{half_life:.1f} days (Long Carryover)", 
                     "Carryover memory is unusually long for a digital performance channel. Check for multi-week seasonality."))
  else:
    callouts.append(("✅", "Adstock Carryover", f"θ = {theta_val:.3f} (Half-Life: {half_life:.2f} days)", "Carryover retention is within standard digital decay windows."))

  # --- Check 5: Causal Experiment Alignment ---
  if experiments and len(experiments) > 0 and hasattr(model, "calibration_experiments") and model.calibration_experiments:
    val = model.validate_experiments(spend_is_raw=True, verbose=False)
    if val["verdict"] == "EXCELLENT":
      callouts.append(("✅", "Causal Experiment Alignment", "EXCELLENT (95% CI Covered)", "Observational curve perfectly replicates experimental ground truth."))
    elif val["verdict"] == "ALIGNED":
      callouts.append(("🟡", "Causal Experiment Alignment", f"ALIGNED (MAPE: {val['mape']:.1f}%)", "Curve is directionally consistent with lift studies."))
    else:
      warnings_count += 1
      callouts.append(("⚠️", "Causal Experiment Alignment", f"MISALIGNED (MAPE: {val['mape']:.1f}%)", 
                       "Significant divergence between observational curve and causal test. Use Bayesian calibration."))

  # --- Print Formatted Health-Check Report ---
  print("\n" + "=" * 90)
  print(f"  🩺 PRACTITIONER MODEL HEALTH CHECK & DIAGNOSTIC CALLOUTS: [{model.channel_name}]")
  print("=" * 90)
  status_banner = "✅ MODEL PASSED ALL DIAGNOSTIC GATES" if warnings_count == 0 else f"⚠️ ATTENTION: {warnings_count} DIAGNOSTIC WARNING(S) FLAGGED"
  print(f"  Overall Health Status: {status_banner}\n")
  
  for icon, check_name, val_str, guidance in callouts:
    print(f"  {icon} {check_name:<34} : {val_str}")
    print(f"     ↳ Guidance: {guidance}\n")
  print("=" * 90)

run_fit_diagnostics(model, metrics, spends, returns, EXPERIMENTS)

## 6. Strategic Tipping Points & Budget Decision Framework

With a validated, diagnostically verified response curve, we identify key operational spend thresholds:

### 1. The Three Critical Boundaries:
*   **Peak Efficiency Point ($f''(x) = 0$):** Inflection point where incremental unit acquisition cost is lowest. Spend at least this amount to exit the inefficient warm-up phase.
*   **Stop Scaling Point ($f'(x) = \text{Target mROAS}$):** The spend ceiling where the marginal return of the next dollar drops below your hurdle rate ($\text{Target mROAS} = 1.25$).
*   **Optimal Scaling Zone:** The high-efficiency growth corridor between Peak Efficiency and Stop Scaling.

### 2. Predictive Uncertainty Intervals:
Using the Delta Method (or Bayesian posterior sampling), we compute rigorous 95% Confidence Intervals for both expected incremental return and marginal efficiency at any target spend level.

In [ ]:
# ------------------------------------------------------------------------------
# 6. TIPPING POINTS & BUDGET DECISION FRAMEWORK
# ------------------------------------------------------------------------------

# Extract key boundaries
min_eff_spend = model.get_minimal_marginal_cost_point()
max_prof_spend = model.get_diminishing_returns_point(target_mroas=TARGET_MROAS, warn_unreachable=False)
opt_window = model.get_optimal_scaling_window(target_mroas=TARGET_MROAS)

min_eff_str = f"${min_eff_spend:,.2f}" if min_eff_spend is not None else "N/A"
max_prof_str = f"${max_prof_spend:,.2f}" if max_prof_spend is not None else f"Unreachable (Max mROAS < {TARGET_MROAS})"
opt_window_str = f"${opt_window[0]:,.2f} to ${opt_window[1]:,.2f}" if opt_window[1] is not None else f"${opt_window[0]:,.2f} to N/A"

print("\n" + "=" * 70)
print(f"  STRATEGIC TIPPING POINTS: {CHANNEL_NAME}")
print("=" * 70)
print(f"  • Minimal Marginal Cost Point (Peak Efficiency): {min_eff_str}")
print(f"  • Point of Diminishing Returns (Target mROAS {TARGET_MROAS}): {max_prof_str}")
print(f"  • Optimal Scaling Zone:                           {opt_window_str}")
print("=" * 70 + "\n")

# Evaluate current operating budget
model.evaluate_current_budget(current_spend=CURRENT_DAILY_SPEND, target_mroas=TARGET_MROAS)

# Scenario Forecast Table with 95% Confidence Intervals
spend_scenarios = [min_eff_spend, CURRENT_DAILY_SPEND]
if max_prof_spend is not None:
  spend_scenarios.extend([max_prof_spend, max_prof_spend * 1.25])
else:
  spend_scenarios.extend([CURRENT_DAILY_SPEND * 1.5, CURRENT_DAILY_SPEND * 2.0])

scenario_rows = []
for s_val in spend_scenarios:
  if s_val is None or np.isnan(s_val):
    continue
  ret_pt, ret_low, ret_high = model.predict_incremental_return(s_val, return_interval=True, confidence_level=0.95)
  mroas_pt, mroas_low, mroas_high = model.predict_marginal_return(s_val, return_interval=True, confidence_level=0.95)
  
  if min_eff_spend is not None and s_val < min_eff_spend:
    status = "Warm-up"
  elif max_prof_spend is not None and s_val > max_prof_spend:
    status = "Over-Saturated"
  else:
    status = "Optimal Zone"
    
  scenario_rows.append({
      "Spend": f"${s_val:,.0f}",
      "Expected Return": f"${ret_pt:,.0f}",
      "Return 95% CI": f"[${ret_low:,.0f}, ${ret_high:,.0f}]",
      "mROAS": f"{mroas_pt:.2f}",
      "mROAS 95% CI": f"[{mroas_low:.2f}, {mroas_high:.2f}]",
      "Scaling Status": status
  })

df_scenarios = pd.DataFrame(scenario_rows)
print("\n--- Scenario Planning & Forecast Matrix ---")
df_scenarios

## 7. Visualizations & Matplotlib Customization

A core requirement for media practitioners is translating mathematical curves into executive-ready assets for budget reviews.

### Two-Step Visualization Pattern:
1. **Module-Based Baseline:** Call `model.plot_response_curve(show=False)` to generate the standard dual-axis visualization.
2. **Post-Hoc Matplotlib Customization:** Intercept the returned `matplotlib.figure.Figure` and its `Axes` (`fig.axes[0]` for Return, `fig.axes[1]` for Marginal ROAS). Inject custom executive callout boxes, directional arrows, branding, or custom hurdle rate overlays before saving.

In [ ]:
# ------------------------------------------------------------------------------
# 7.1 & 7.2: MODULE VISUALIZATION & INTERCEPTING THE MATPLOTLIB FIGURE
# ------------------------------------------------------------------------------

# Generate the module-based visual with show=False to intercept the Figure object
fig = model.plot_response_curve(
    target_mroas=TARGET_MROAS,
    current_spend=CURRENT_DAILY_SPEND,
    show_intervals=True,
    scatter=(spends, returns),
    show=False  # Crucial: suppresses immediate plt.show() so edits can be applied
)

# Intercept underlying primary and secondary axes
ax1 = fig.axes[0]  # Primary Axis: Incremental Return
ax2 = fig.axes[1]  # Secondary Axis: Marginal ROAS

# ------------------------------------------------------------------------------
# 🎨 PRACTITIONER EDITS: Custom annotations, executive callout box, and arrows
# ------------------------------------------------------------------------------

# 1. Add an Executive Summary Callout Box to the upper left
in_optimal = (min_eff_spend <= CURRENT_DAILY_SPEND <= (max_prof_spend if max_prof_spend else np.inf))
status_str = "OPTIMAL SCALING ZONE" if in_optimal else ("WARMING UP" if CURRENT_DAILY_SPEND < min_eff_spend else "OVER-SATURATED")
cap_str = f"${max_prof_spend:,.0f}/day" if max_prof_spend else "N/A"
headroom_str = f"+${max(0.0, max_prof_spend - CURRENT_DAILY_SPEND):,.0f}/day" if max_prof_spend else "N/A"

callout_text = (
    f"EXECUTIVE SUMMARY:\n"
    f"• Current Spend: ${CURRENT_DAILY_SPEND:,.0f}/day\n"
    f"• Status: {status_str}\n"
    f"• Saturation Cap (mROAS={TARGET_MROAS}): {cap_str}\n"
    f"• Max Headroom: {headroom_str}"
)
ax1.text(
    0.03, 0.93, callout_text,
    transform=ax1.transAxes,
    fontsize=9.5,
    verticalalignment='top',
    bbox=dict(boxstyle='round,pad=0.6', facecolor='#F8F9FA', edgecolor='#DADCE0', alpha=0.95, linewidth=1.2)
)

# 2. Add an arrow annotation pointing directly to the current operating spend point
current_ret = model.predict_incremental_return(CURRENT_DAILY_SPEND)
ax1.annotate(
    f'Current Operating Spend\n(${CURRENT_DAILY_SPEND:,.0f} → ${current_ret:,.0f})',
    xy=(CURRENT_DAILY_SPEND, current_ret),
    xytext=(CURRENT_DAILY_SPEND * 1.15, current_ret * 0.70),
    arrowprops=dict(facecolor='#EA4335', edgecolor='#EA4335', arrowstyle='->', lw=1.5),
    fontsize=9.5, fontweight='bold', color='#C5221F',
    bbox=dict(boxstyle='square,pad=0.3', facecolor='#FCE8E6', edgecolor='#F28B82', alpha=0.9)
)

# 3. Add Custom Confidential / Internal Watermark or Timestamp
fig.text(0.98, 0.02, "Generated by Tipping Point | Marketing Intelligence", 
         ha='right', va='bottom', fontsize=8, color='#80868B', style='italic')

# 4. Optional: Save high-resolution graphic for executive slide decks
# fig.savefig("single_channel_saturation_report.png", dpi=300, bbox_inches='tight')

# Display the finalized, customized figure
plt.show()

### 7.3 Standalone Multi-Panel Diagnostic & Performance Dashboard

For econometric reviews and audits, we generate a 3-panel diagnostic visualization displaying:
1. **Saturation Response Curve** with 95% Confidence Intervals, optimal scaling zones, and historical data points.
2. **Marginal ROAS Trajectory** with hurdle rate threshold and Peak Efficiency point.
3. **Residual Error Diagnostic** showing predicted values vs error residuals within $\pm 1$ RMSE bands.

In [ ]:
# ------------------------------------------------------------------------------
# 7.3: 3-PANEL EXECUTIVE DIAGNOSTIC DASHBOARD
# ------------------------------------------------------------------------------

# Compute grid predictions
ref_max_spend = max_prof_spend if (max_prof_spend is not None and not np.isnan(max_prof_spend)) else np.max(spends)
max_x = max(ref_max_spend * 1.4, np.max(spends) * 1.1)
x_grid = np.linspace(100.0, max_x, 300)

y_grid, y_grid_low, y_grid_high = model.predict_incremental_return(x_grid, return_interval=True, confidence_level=0.95)
mroas_grid, mroas_low, mroas_high = model.predict_marginal_return(x_grid, return_interval=True, confidence_level=0.95)

# Historical predictions & residuals
spends_adstocked = model.adstock_spend(spends)
y_pred_hist = model.predict_incremental_return(spends_adstocked, include_baseline=True)
residuals = returns - y_pred_hist

# Create 3-panel figure
fig, (ax_top, ax_mid, ax_bot) = plt.subplots(3, 1, figsize=(12, 14), facecolor='white')
fig.subplots_adjust(hspace=0.35)

# --- Panel 1: Response Curve ---
ax_top.set_facecolor('white')
ax_top.plot(x_grid, y_grid, color='#4285F4', lw=3.0, label='Fitted Response Curve', zorder=3)
ax_top.fill_between(x_grid, y_grid_low, y_grid_high, color='#4285F4', alpha=0.15, label='95% Confidence Interval', zorder=2)
ax_top.scatter(spends_adstocked, returns, color='#5F6368', alpha=0.35, s=30, label='Historical Data (Adstocked)', zorder=1)

if min_eff_spend is not None and max_prof_spend is not None and max_prof_spend > min_eff_spend:
  ax_top.axvspan(min_eff_spend, max_prof_spend, color='#34A853', alpha=0.10, label='Optimal Scaling Zone', zorder=0)

ax_top.axvline(CURRENT_DAILY_SPEND, color='#EA4335', ls='--', lw=1.5, label=f'Current Spend (${CURRENT_DAILY_SPEND:,.0f})')
ax_top.set_title(f"A. Media Saturation & Scaling Boundaries: {CHANNEL_NAME}", fontsize=13, fontweight='bold', loc='left', pad=10)
ax_top.set_ylabel("Incremental Return ($)", fontsize=10, color='#5F6368')
ax_top.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, p: f"${x*1e-3:g}k" if x >= 1e3 else f"${x:g}"))
ax_top.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, p: f"${x*1e-3:g}k" if x >= 1e3 else f"${x:g}"))
ax_top.legend(loc='lower right', frameon=True, facecolor='white', framealpha=0.9, fontsize=9)
ax_top.grid(True, alpha=0.15)

# --- Panel 2: Marginal Efficiency & Unit Economics ---
ax_mid.set_facecolor('white')
ax_mid.plot(x_grid, mroas_grid, color='#188038', lw=2.5, label='Marginal ROAS (mROAS)', zorder=3)
ax_mid.fill_between(x_grid, mroas_low, mroas_high, color='#188038', alpha=0.12, zorder=2)
ax_mid.axhline(TARGET_MROAS, color='#EA4335', ls=':', lw=1.5, label=f'Target mROAS Hurdle ({TARGET_MROAS:.2f})')

if min_eff_spend is not None and min_eff_spend > 0:
  ax_mid.scatter([min_eff_spend], [model.predict_marginal_return(min_eff_spend)], color='#FBBC04', s=80, edgecolors='#5F6368', zorder=4, label=f'Peak Efficiency (${min_eff_spend:,.0f})')

ax_mid.set_title("B. Marginal Efficiency Curve (mROAS vs Spend)", fontsize=13, fontweight='bold', loc='left', pad=10)
ax_mid.set_ylabel("mROAS ($ Return / $ Spend)", fontsize=10, color='#5F6368')
ax_mid.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, p: f"${x*1e-3:g}k" if x >= 1e3 else f"${x:g}"))
ax_mid.set_ylim(bottom=0)
ax_mid.legend(loc='upper right', frameon=True, facecolor='white', framealpha=0.9, fontsize=9)
ax_mid.grid(True, alpha=0.15)

# --- Panel 3: Goodness-of-Fit Residuals ---
ax_bot.set_facecolor('white')
ax_bot.scatter(y_pred_hist, residuals, color='#4285F4', alpha=0.5, s=35, edgecolors='white', lw=0.5)
ax_bot.axhline(0, color='#EA4335', ls='--', lw=1.5)
ax_bot.axhspan(-metrics["rmse"], metrics["rmse"], color='#5F6368', alpha=0.08, label=f'±1 RMSE Band (±${metrics["rmse"]:,.0f})')
ax_bot.set_title("C. Goodness-of-Fit: Residual Error Diagnostic (Actual - Predicted vs Fitted)", fontsize=13, fontweight='bold', loc='left', pad=10)
ax_bot.set_xlabel("Fitted Return ($)", fontsize=10, color='#5F6368')
ax_bot.set_ylabel("Residual Error ($)", fontsize=10, color='#5F6368')
ax_bot.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, p: f"${x*1e-3:g}k" if x >= 1e3 else f"${x:g}"))
ax_bot.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, p: f"${x*1e-3:g}k" if x >= 1e3 else f"${x:g}"))
ax_bot.legend(loc='upper left', frameon=True, facecolor='white', framealpha=0.9, fontsize=9)
ax_bot.grid(True, alpha=0.15)

plt.show()

## 8. Forward-Looking Capacity Projection & Benchmark Modifiers

Historical media response curves measure **what happened historically**, which is fundamentally bounded by the advertiser's historical activation tactics and channel maturity.

### The Observational Saturation Bottleneck
*   **Lower-Funnel Truncation:** Advertisers running solely lower-funnel tactics (e.g., action-oriented direct-response video) face early saturation because they repeatedly engage high-intent users without replenishing the upstream prospect pool.
*   **Audience Under-Penetration:** Low Reach & Frequency (R&F) relative to the addressable market compresses both the asymptotic return ceiling ($\beta$) and the half-saturation spend point ($K$).

Treating an observational curve as an immutable ceiling leads to premature budget capping and missed growth opportunities.

---

### The Damped Multiplier Formulation
`tippingpoint.capacity` formalizes forward-looking capacity projection via distilled, empirical multipliers applied to the base curve parameters:
$$\beta_{\text{proj}} = M_{\beta} \cdot \beta, \qquad K_{\text{proj}} = M_K \cdot K$$

Where multipliers are synthesized across two core growth levers:
1.  **Funnel / Product Evolution ($\Delta M_{\text{product}}$):** Unlocking upper- and mid-funnel formats (e.g., migrating from *lower-funnel only* $\to$ *full-funnel*).
2.  **Reach & Frequency Expansion ($\Delta M_{\text{rf}}$):** Expanding reach penetration across the Total Addressable Market (TAM) and normalizing frequency to effective levels.
3.  **Sub-Additive Synergy Synthesis:** Because audience expansion and multi-format storytelling reinforce each other sub-additively, we apply an empirical damping factor $\rho_{\text{synergy}} \in (0, 1]$ (default $0.85$):
$$M = 1.0 + \rho_{\text{synergy}} \cdot \Big((M_{\text{rf}} - 1.0) + (M_{\text{product}} - 1.0)\Big)$$

---

### Empirical Benchmark Coefficients
Practitioners can supply vertical benchmark coefficients as:
*   **Pre-compiled research benchmarks:** (e.g., `research.tls_benchmarks` developed from Nielsen meta-analyses, Meridian MMMs, and internal conversion panels).
*   **Enterprise / Vertical JSON configs:** Standardized multiplier files distributed across your organization.
*   **Custom in-memory dictionaries:** Advertiser-specific test learnings and media planning guidelines.

In [ ]:
# ------------------------------------------------------------------------------
# 8. FORWARD-LOOKING CAPACITY PROJECTION WITH BENCHMARK COEFFICIENTS
# ------------------------------------------------------------------------------
from tippingpoint.capacity import CapacityProjector, ProjectedReturnCurve

# 1. Load or Define Vertical Benchmark Coefficients
# We attempt to import empirical research benchmarks from the research directory,
# falling back to an inline vertical configuration dictionary if running standalone.
try:
  from research.tls_benchmarks import TLS_VERTICAL_BENCHMARKS
  benchmark_source = "research.tls_benchmarks (Empirical Multipliers)"
  vertical_benchmarks = TLS_VERTICAL_BENCHMARKS
except ImportError:
  benchmark_source = "Inline Vertical Benchmark Config"
  vertical_benchmarks = {
      "tech_b2b": {
          "lower_only": {"m_beta": 1.00, "m_k": 1.00},
          "mid_lower": {"m_beta": 1.25, "m_k": 1.35},
          "upper_lower": {"m_beta": 1.38, "m_k": 1.55},
          "full_funnel": {"m_beta": 1.55, "m_k": 1.85},
          "rf_targets": {"target_frequency": 2.2, "target_reach_penetration": 0.60},
      },
      "apps_platforms": {
          "lower_only": {"m_beta": 1.00, "m_k": 1.00},
          "mid_lower": {"m_beta": 1.35, "m_k": 1.50},
          "upper_lower": {"m_beta": 1.55, "m_k": 1.85},
          "full_funnel": {"m_beta": 1.80, "m_k": 2.35},
          "rf_targets": {"target_frequency": 3.0, "target_reach_penetration": 0.45},
      },
  }

selected_vertical = "tech_b2b"

print("\n" + "=" * 75)
print(f"  CAPACITY PROJECTION: {CHANNEL_NAME} [{selected_vertical.upper()}]")
print(f"  Benchmark Source: {benchmark_source}")
print("=" * 75)

# 2. Forward-Looking Capacity Projection via model.project_capacity()
# Scenario: Transitioning from lower-funnel only to full-funnel video
# while scaling TAM reach from 25% to 55% at target frequency 2.2.
proj_curve = model.project_capacity(
    vertical=selected_vertical,
    vertical_multipliers=vertical_benchmarks,
    current_funnel="lower_only",
    target_funnel="full_funnel",
    tam_size=5_000_000,
    current_reach=1_250_000,          # Current 25% TAM reach penetration
    target_reach_penetration=0.55,    # Target 55% TAM reach penetration
    current_frequency=1.6,            # Current average frequency
    target_frequency=2.2,             # Target frequency from vertical benchmarks
    synergy_damping=0.85,             # Sub-additive interaction factor (rho)
    channel_name=f"{CHANNEL_NAME} (Full-Funnel Projected)"
)

# 3. Multiplier Breakdown & Driver Diagnostics
mult_summary = proj_curve.multipliers.summary()
p_comp = mult_summary["product_component"]
rf_comp = mult_summary["rf_component"]

print(f"\n📈 Multiplier Synthesis (Synergy Damping rho = {mult_summary['synergy_damping']:.2f}):")
print(f"   • Funnel Evolution ({p_comp['details']['current_funnel']} -> {p_comp['details']['target_funnel']}):")
print(f"       M_beta_product = {p_comp['m_beta']:.2f}x | M_K_product = {p_comp['m_k']:.2f}x")
print(f"   • Audience Penetration (Reach: {rf_comp['details']['rho_curr']*100:.0f}% -> {rf_comp['details']['rho_target']*100:.0f}%, Freq: {rf_comp['details']['current_frequency']:.1f} -> {rf_comp['details']['target_frequency']:.1f}):")
print(f"       M_beta_rf      = {rf_comp['m_beta']:.2f}x | M_K_rf      = {rf_comp['m_k']:.2f}x")
print(f"   • Effective Combined Multipliers:")
print(f"       Total M_beta   = {mult_summary['m_beta_total']:.2f}x (Saturation Ceiling)")
print(f"       Total M_K      = {mult_summary['m_k_total']:.2f}x (Half-Saturation Spend Dilation)")

# 4. Strategic Headroom & Budget Unlock Report
eval_rate = TARGET_MROAS
proj_peak_spend = proj_curve.get_minimal_marginal_cost_point()
proj_peak_mroas = proj_curve.predict_marginal_return(proj_peak_spend) if proj_peak_spend else 0.0
if proj_peak_mroas < eval_rate:
  print(f"\nℹ️  Notice: Target hurdle rate ({eval_rate:.2f}) exceeds projected peak mROAS ({proj_peak_mroas:.2f}).")
  print(f"   Evaluating headroom at breakeven return (mROAS = 1.00):")
  eval_rate = 1.0

headroom_report = proj_curve.evaluate_unlocked_headroom(
    target_mroas=eval_rate,
    current_spend=CURRENT_DAILY_SPEND,
    verbose=True
)

# 5. Dual-Curve Comparative Visualization
fig_proj = proj_curve.plot_curve_comparison(
    target_mroas=eval_rate,
    current_spend=CURRENT_DAILY_SPEND,
    show=True
)

## 9. Practitioner Governance Checklist & Next Steps

Before presenting response curves or executing media budget changes, verify the following governance quality gates:

- [ ] **1. Data Alignment & Quality:** Spend and return series are aligned on matching calendar timestamps without missing intervals.
- [ ] **2. Organic Baseline Accounting:** Verified whether baseline return ($\beta_0$) is properly estimated or if returns represent pure incremental media return.
- [ ] **3. Causal Incrementality Validation & Calibration:** Validated that observational predictions fall within the 95% CI of recent lift tests (`model.validate_experiments`).
- [ ] **4. Statistical Fit Diagnostic:** $R^2 \ge 0.60$, $\text{MAPE} \le 25\%$, and residual errors pass unbiasedness tests without severe heteroscedasticity.
- [ ] **5. Extrapolation Guardrails:** Recommended scaling budget is supported by historical observations and does not exceed $2 \times K$.
- [ ] **6. Forward-Looking Capacity Verification:** When applying benchmark multipliers (`model.project_capacity`), ensure vertical benchmarks align with advertiser business model, R&F headroom is grounded in addressable TAM, and sub-additive synergy damping ($\rho \le 0.85$) is applied.
- [ ] **7. Interactive Web Dashboard:** For live stakeholder walkthroughs and parameter scenario exploration, launch the built-in UI:
  ```python
  # Uncomment to launch the interactive Streamlit dashboard:
  # model.launch_dashboard()
  ```